# 07 — Stage-2 候选→最终漏斗审计

当 `training.stage2_candidate_count > final_feature_count` 时，Stage-1 把候选池写到 `selected_features/v1_auto.txt`，Stage-2 跑一次廉价探索 XGBoost 用配置的 ranker（gain / stability / shap / …）打分后再砍到 `final_feature_count`。

本 notebook 消费产物（不重训）：
- `models/<run>/exploratory_importance.csv` — 候选 score + `kept` + `ranking_method`
- `models/<run>/pruned_features.txt` — 漏斗后留下的 `final_feature_count` 个特征（base + indicator）
- `models/<run>/run_manifest.json::feature_funnel` — 漏斗参数 fingerprint

回答三个问题：
1. 切线在哪？top-k 的 score 分布是平滑下降还是有断档？
2. 被砍掉的特征长什么样？（IV / family / semantic 维度上是不是被低估了）
3. 不同 ranker 的结果稳不稳？（同一个 run 跑两次 stability vs shap，重叠率多少）

In [ ]:
import sys
sys.path.insert(0, '../src')
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wdm.config import load_config
from wdm.utils.paths import report_dir

PRODUCT = 'hzz_day'        # hzz_day 是目前唯一开了漏斗的产品（home_credit 走 legacy 直通路径）
RUN_ID  = 'smoke02'        # models/<PRODUCT>/<RUN_ID>/
RUN_ID_ALT = None          # 设成另一个 run_id 即可做跨 ranker / 跨参数对比

cfg = load_config(PRODUCT)
rd_run = Path(cfg['_repo_root']) / 'artifacts' / PRODUCT / 'models' / RUN_ID
rd_report = report_dir(cfg)
print('run dir   :', rd_run)
print('report dir:', rd_report)

In [ ]:
# Manifest 上的 funnel fingerprint —— 训练时实际生效的参数
manifest = json.loads((rd_run / 'run_manifest.json').read_text(encoding='utf-8'))
funnel = manifest.get('feature_funnel') or {}
print('feature_funnel:')
for k, v in funnel.items():
    print(f'  {k}: {v}')
print()
print(f'n_features_base       : {manifest["n_features_base"]}')
print(f'n_features_indicator  : {manifest["n_features_indicator"]}')
print(f'n_features_total      : {manifest["n_features_total"]}')
candidate_n = funnel.get('stage2_candidate_count')
final_n = funnel.get('final_feature_count')
if candidate_n is None or candidate_n <= final_n:
    print('\n⚠ 漏斗未启用（stage2_candidate_count 为空或 ≤ final_feature_count），')
    print('  本 run 走的是 legacy 直通路径，下面的 §1–§3 没有数据可看。')
    print('  把 configs/products/{}.yaml 里的 stage2_candidate_count 设大一点再训一次即可。'.format(PRODUCT))

## 1. 切线诊断 —— top-k 的 score 分布

In [ ]:
imp_path = rd_run / 'exploratory_importance.csv'
if not imp_path.is_file():
    print('未找到 exploratory_importance.csv —— 该 run 没经过漏斗（pool ≤ final_feature_count）。')
    imp = None
else:
    imp = pd.read_csv(imp_path).sort_values('score', ascending=False).reset_index(drop=True)
    imp['rank'] = imp.index + 1
    method = imp['ranking_method'].iloc[0] if 'ranking_method' in imp.columns else funnel.get('ranking_method', '?')
    n_kept = int(imp['kept'].sum())
    n_total = len(imp)
    print(f'ranker     : {method}')
    print(f'candidates : {n_total}')
    print(f'kept       : {n_kept} ({n_kept / n_total:.0%})')
    imp.head(10)

In [ ]:
if imp is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    # 左：score vs rank；切线高亮
    axes[0].plot(imp['rank'], imp['score'], lw=1.3, color='steelblue')
    cut_rank = int(imp['kept'].sum())
    axes[0].axvline(cut_rank, ls='--', color='crimson', alpha=0.7,
                    label=f'cut @ rank {cut_rank}')
    axes[0].set_xlabel('rank (排序后)')
    axes[0].set_ylabel(f'score ({imp["ranking_method"].iloc[0]})')
    axes[0].set_title('1.1 score vs rank —— 切线两侧是否有断档')
    axes[0].grid(alpha=0.3); axes[0].legend()
    # 右：log-score 直方，看分布形态
    pos = imp.loc[imp['score'] > 0, 'score']
    axes[1].hist(np.log10(pos + 1e-12), bins=50, color='gray', alpha=0.75)
    axes[1].set_xlabel('log10(score)')
    axes[1].set_ylabel('count')
    axes[1].set_title('1.2 score 分布 —— 长尾还是双峰')
    axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    
    # 切线邻域：±10 的特征明细
    edge = imp.iloc[max(0, cut_rank - 10): cut_rank + 10][['rank', 'feature', 'score', 'kept']]
    print('\n切线邻域（±10）—— 这些是被「微弱差距砍掉」或「擦边留下」的：')
    print(edge.to_string(index=False))

## 2. 被砍掉的特征 —— Stage-1 维度画像

把 `exploratory_importance.csv` 跟 Stage-1 的 `summary.csv` join 起来，看丢掉的 600 个候选在 IV / window / semantic_group 维度上是否系统性偏离 kept 集合。如果 dropped 集中在某个 semantic group 或某种 window，说明 ranker 跟 Stage-1 的家族策略不一致，值得排查。

In [ ]:
if imp is None:
    print('skip')
else:
    summary = pd.read_csv(rd_report / 'summary.csv')
    keep_cols = [c for c in ['feature', 'iv', 'missing_rate', 'lift_at_k',
                              'window', 'semantic_group', 'family_id',
                              'auto_keep', 'rank_score']
                  if c in summary.columns]
    joined = imp.merge(summary[keep_cols], on='feature', how='left')
    print('joined:', joined.shape, '（NaN 行 = exploratory_importance 里有但 Stage-1 summary 找不到的）')
    joined.head()

In [ ]:
if imp is None:
    pass
elif 'iv' in joined.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    for ax, col, title in [
        (axes[0], 'iv', '2.1 IV 分布：kept vs dropped'),
        (axes[1], 'missing_rate', '2.2 缺失率分布：kept vs dropped'),
    ]:
        if col not in joined.columns:
            ax.set_visible(False); continue
        kept_v = joined.loc[joined['kept'], col].dropna()
        drop_v = joined.loc[~joined['kept'], col].dropna()
        bins = np.linspace(0, max(kept_v.max(), drop_v.max(), 1e-6), 40)
        ax.hist(drop_v, bins=bins, alpha=0.55, label=f'dropped (n={len(drop_v)})', color='crimson')
        ax.hist(kept_v, bins=bins, alpha=0.55, label=f'kept    (n={len(kept_v)})', color='steelblue')
        ax.set_xlabel(col); ax.set_ylabel('count')
        ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
# 2.3 按 semantic_group / window 看 kept 比例 —— 哪个簇被漏斗优待了
if imp is None:
    pass
else:
    for col in ['semantic_group', 'window']:
        if col not in joined.columns:
            continue
        agg = joined.dropna(subset=[col]).groupby(col).agg(
            n_candidate=('feature', 'count'),
            n_kept=('kept', 'sum'),
        )
        agg['kept_rate'] = agg['n_kept'] / agg['n_candidate']
        agg = agg.sort_values('kept_rate', ascending=False)
        print(f'\n--- kept rate by {col} ---')
        print(agg.to_string())

## 3. 跨 run 对比 —— 不同 ranker 是否选出相似的特征集

稳定性是漏斗的重要属性：不同 ranker（或同一个 ranker 在不同 seed）应当大致选出相似的 final 200。重叠率低 → ranker 受随机性 / 候选噪声主导，需要换 stability 变体或加大 `n_seeds`。

把 `RUN_ID_ALT` 设成另一个开了漏斗的 run（比如 `ranking_method=stability` 跟 `shap` 对比）即可。

In [ ]:
if imp is None or RUN_ID_ALT is None:
    print('skip —— RUN_ID_ALT 未设，或当前 run 没经过漏斗。')
else:
    rd_alt = Path(cfg['_repo_root']) / 'artifacts' / PRODUCT / 'models' / RUN_ID_ALT
    imp_alt = pd.read_csv(rd_alt / 'exploratory_importance.csv')
    kept_a = set(imp.loc[imp['kept'], 'feature'])
    kept_b = set(imp_alt.loc[imp_alt['kept'], 'feature'])
    inter = kept_a & kept_b
    union = kept_a | kept_b
    method_a = imp['ranking_method'].iloc[0]
    method_b = imp_alt['ranking_method'].iloc[0]
    print(f'{RUN_ID:>20s} ({method_a:>20s}): {len(kept_a)} kept')
    print(f'{RUN_ID_ALT:>20s} ({method_b:>20s}): {len(kept_b)} kept')
    print(f'{"intersection":>20s}                       : {len(inter)}')
    print(f'{"jaccard":>20s}                       : {len(inter) / len(union):.2%}')
    print(f'{"only in A":>20s}                       : {len(kept_a - kept_b)}')
    print(f'{"only in B":>20s}                       : {len(kept_b - kept_a)}')

## 结论

- **切线健康度**：score 在 cut_rank 两侧应当连续平滑。若出现明显断档（score 突然塌陷），说明候选池里有一批「显著强」+ 一批「显著弱」，ranker 工作正常；若曲线平直，说明 final 200 跟邻近 600 的差距小，调高 `final_feature_count` 或换更稳定的 ranker 都可能有改善空间。
- **dropped 画像**：被砍特征若集中在某个 `semantic_group` / `window`，要回头看 Stage-1 家族政策（`group_max_keep`、`family_policy`）是不是把同质特征塞进了候选池。
- **跨 ranker 稳定性**：经验阈值——stability vs shap 的 jaccard ≥ 0.7 算稳；< 0.5 说明信号噪声比偏低，最稳妥的做法是切到 `permutation_stability` 或 `shap_stability`，并把 `n_seeds` 拉到 5+。